# 🐖 12 | Eficiencia computacional de YOLO11n-seg y Mask R-CNN

**RECURSO: GPU T4** (o GPU claramente identificada). **No entrena.** Reutiliza los mejores pesos de los notebooks 03 y 06. Mide tiempos y memoria de las dos canalizaciones completas bajo un único entorno Colab.

**Diseño del benchmark:** **72 imágenes de validación**, 18 por cada una de sus cuatro fechas, seleccionadas con semilla 2026; 5 iteraciones de precalentamiento por modelo; **2 repeticiones** sobre las mismas imágenes; sincronización CUDA. No utiliza test para ajustar ni elegir configuraciones. No requiere ejecutar el Notebook 11 primero.

**Comparabilidad:** se evalúan las canalizaciones originales de cada modelo (preprocesamiento, inferencia, postprocesamiento y transferencia de máscaras a CPU) con una misma GPU y las mismas imágenes. No se igualan tamaños de entrada, arquitecturas ni presupuestos de entrenamiento, por lo que la diferencia de latencia es **del sistema de inferencia**, no velocidad intrínseca universal de la arquitectura. El consumo de VRAM se reporta como memoria máxima **asignada por PyTorch** (no memoria física total ni energía).

**Salida:** Excel de latencias por imagen/repetición, métricas descriptivas, gráfico, reporte JSON y ZIP. El benchmark es experimental: no mezclar resultados de GPU distinta, software distinto o muestra distinta.

## 1. Comprobar GPU ANTES de descargar dependencias

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Notebook 12 requiere GPU: Entorno de ejecución > Cambiar tipo de entorno de ejecución > T4 GPU')
print('✅ GPU activa:',torch.cuda.get_device_name(0))


In [ ]:
%pip -q install "ultralytics>=8.3,<9" "pyyaml>=6,<7" "pandas>=2,<3" openpyxl pillow matplotlib


## 2. Rutas, pesos originales y selección reproducible de validación

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json,random,time,zipfile,sys,platform,gc
import numpy as np,pandas as pd,yaml,torch,torchvision
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO
from torchvision.transforms.functional import to_tensor
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
DATA=ROOT/'01_Datasets'/'FaroPigSeg'
RES=ROOT/'03_Resultados';REPORTS=RES/'05_Reportes';METRICS=RES/'03_Metricas'
FIGURES=RES/'04_Figuras';EXPORT=ROOT/'05_Exportaciones_ZIP'
RUN=RES/'06_Entrenamiento'
YAML=DATA/'split_fecha_v1'/'data_fecha.yaml'
YOLO_W=RUN/'YOLO11n_fecha_v1'/'weights'/'best.pt'
MASK_W=RUN/'MaskRCNN_fecha_v1'/'maskrcnn_best_model.pt'
PROTO09=REPORTS/'CERDOS_09_COCO_Armonizado'/'CERDOS_09_Protocolo.json'
for p in [YAML,YOLO_W,MASK_W,PROTO09]:
    if not p.is_file():raise FileNotFoundError(f'Falta {p}; no repetir entrenamiento')
for p in [REPORTS,METRICS,FIGURES,EXPORT]:p.mkdir(parents=True,exist_ok=True)
def sha256_file(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()
old=json.loads(PROTO09.read_text(encoding='utf-8'))
assert sha256_file(YOLO_W)==old['yolo_sha256'],'Pesos YOLO distintos de la evaluación 09'
assert sha256_file(MASK_W)==old['mask_sha256'],'Pesos Mask R-CNN distintos de la evaluación 09'
assert sha256_file(YAML)==old['yaml_sha256'],'YAML diferente del 09'
conf=yaml.safe_load(YAML.read_text(encoding='utf-8'))
val_list=Path(conf['val'])
assert val_list.is_file(),'Lista de validación no encontrada'
val_images=[Path(x.strip()) for x in val_list.read_text(encoding='utf-8').splitlines() if x.strip()]
assert len(val_images)==196 and len(set(map(str,val_images)))==196 and all(x.is_file() for x in val_images)
dates=sorted({p.name[:10] for p in val_images})
assert len(dates)==4,'Se esperaban cuatro fechas de validación'
rng=random.Random(2026)
sampled=[]
for date in dates:
    candidates=sorted(p for p in val_images if p.name.startswith(date))
    assert len(candidates)>=18,'No hay 18 imágenes de validación para una de las fechas'
    sampled.extend(rng.sample(candidates,18))
sampled=sorted(sampled,key=lambda p:(p.name[:10],p.name))
assert len(sampled)==72
GPU=torch.cuda.get_device_name(0)
print('✅ Muestra congelada:',len(sampled),'imágenes de validación de',len(dates),'fechas')
print('✅ GPU:',GPU,'| GPU de la sesión real; si es distinta de T4, quedará registrado')
print('✅ No se accede al conjunto test ni se modifica ningún peso.')


## 3. Protocolo de benchmark y datos precargados en RAM
 Se excluye el tiempo de lectura desde Drive para medir el comportamiento del sistema de inferencia sobre las mismas imágenes. Se registra la resolución individual y el costo de transferir las máscaras resultantes a CPU.

In [ ]:
EXCEL=METRICS/'CERDOS_12_Eficiencia_Computacional.xlsx'
FIG=FIGURES/'CERDOS_12_Latencia_Arquitecturas.png'
JREPORT=REPORTS/'CERDOS_12_Eficiencia_Computacional.json'
PFILE=REPORTS/'CERDOS_12_Protocolo.json'
PROTO={'pipeline':'CERDOS_12_inferencia_v1','dataset':'FaroPigSeg_val_72',
       'lista_val_sha256':sha256_file(val_list),
       'archivos_muestra':[p.name for p in sampled],
       'gpu_real':GPU,'gpu_compute_capability':list(torch.cuda.get_device_capability(0)),
       'torch':torch.__version__,'torchvision':torchvision.__version__,
       'ultralytics':__import__('ultralytics').__version__,
       'semilla':2026,'fechas_validacion':dates,'imagenes_por_fecha':18,
       'warmup':5,'repeticiones':2,'fp16':False,
       'yolo_imgsz':old['yolo_imgsz'],'yolo_conf':old['yolo_conf'],
       'yolo_iou_nms':old['yolo_iou_nms'],'yolo_retina_masks':old['yolo_retina_masks'],
       'maskrcnn_min_size':512,'maskrcnn_max_size':768,
       'mask_score_thresh':old['mask_score_thresh'],'mask_bin_thresh':old['mask_bin_thresh'],
       'yolo_weights_sha256':old['yolo_sha256'],'mask_weights_sha256':old['mask_sha256'],
       'ram_preload':True,'cuda_synchronize':True,
       'timing':'preprocesado + red + postprocesado + máscaras CPU; lectura de disco excluida',
       'memoria':'max_memory_allocated de CUDA/PyTorch, modelo y ejecución incluidos'}
if PFILE.exists():
    if json.loads(PFILE.read_text(encoding='utf-8'))!=PROTO:
        raise RuntimeError('Benchmark 12 anterior con otro protocolo o GPU: no sobrescribir. Conserva el resultado anterior.')
else:
    if EXCEL.exists() or JREPORT.exists():
        raise RuntimeError('Resultados previos del 12 sin protocolo. Revisa Drive, no sobrescribir.')
    PFILE.write_text(json.dumps(PROTO,ensure_ascii=False,indent=2),encoding='utf-8')
RGB=[]
for i,p in enumerate(sampled,start=1):
    with Image.open(p) as im:RGB.append(np.asarray(im.convert('RGB')).copy())
    if i%18==0:print('Imágenes en RAM:',i,'/',len(sampled))
print('✅ RAM con 72 imágenes; el tiempo medido no incluye operaciones de lectura de Drive')


## 4. Ejecutar mediciones YOLO11n-seg (una única arquitectura a la vez)
 Con calentamiento previo y sincronización CUDA; `predict` incluye la decodificación/postproceso Ultralytics. El conteo de máscaras transferidas se registra sin usarse para elegir parámetros.

In [ ]:
def collect_yolo_stats(model,rgb):
    with torch.inference_mode():
        out=model.predict(source=rgb,device=0,imgsz=PROTO['yolo_imgsz'],
            conf=PROTO['yolo_conf'],iou=PROTO['yolo_iou_nms'],
            max_det=old['max_dets'],retina_masks=True,half=False,save=False,verbose=False)[0]
        n=0
        if out.masks is not None and out.masks.data is not None:
            masks=out.masks.data.detach().cpu().numpy()
            n=int(np.count_nonzero(masks.reshape(len(masks),-1).max(axis=1)>0))
        if out.boxes is not None:
            _=out.boxes.conf.detach().cpu().numpy()
        del out
    return n

def timed_model(model,name,fn):
    for i in range(PROTO['warmup']):
        _=fn(model,RGB[i])
    torch.cuda.synchronize(0)
    torch.cuda.reset_peak_memory_stats(0)
    records=[]
    for rep in range(PROTO['repeticiones']):
        for idx,(path,rgb) in enumerate(zip(sampled,RGB),start=1):
            torch.cuda.synchronize(0)
            start=time.perf_counter_ns()
            n=fn(model,rgb)
            torch.cuda.synchronize(0)
            elapsed=(time.perf_counter_ns()-start)/1_000_000
            records.append({'modelo':name,'repeticion':rep+1,'indice':idx,'fecha':path.name[:10],
                'imagen':path.name,'ancho_px':int(rgb.shape[1]),'alto_px':int(rgb.shape[0]),
                'latencia_ms':float(elapsed),'mascaras_devueltas':int(n)})
        print(name,'repetición',rep+1,'/',PROTO['repeticiones'],'lista')
    peak_mb=float(torch.cuda.max_memory_allocated(0)/1024**2)
    return records,peak_mb

if EXCEL.is_file() and JREPORT.is_file():
    print('✅ Benchmark previo íntegro detectado; no se repite inferencia.')
    df_raw=pd.read_excel(EXCEL,sheet_name='Latencias_Individuales')
    memory_peak=json.loads(JREPORT.read_text(encoding='utf-8'))['picos_vram_mib']
else:
    TEMP=REPORTS/'CERDOS_12_Cache_YOLO.json'
    if TEMP.exists():
        cache=json.loads(TEMP.read_text(encoding='utf-8'))
        if cache['protocolo']!=PROTO:raise RuntimeError('Caché YOLO con protocolo distinto')
        yr=cache['datos'];ypeak=float(cache['vram_mib'])
        assert len(yr)==len(sampled)*PROTO['repeticiones'],'Caché incompleta'
        print('✅ YOLO ya medido; se reutiliza su caché sin repetir inferencia')
    else:
        model=YOLO(str(YOLO_W))
        yr,ypeak=timed_model(model,'YOLO11n-seg',collect_yolo_stats)
        del model;gc.collect();torch.cuda.empty_cache()
        TEMP.write_text(json.dumps({'protocolo':PROTO,'datos':yr,'vram_mib':ypeak},ensure_ascii=False),encoding='utf-8')
    df_raw=pd.DataFrame(yr)
    memory_peak={'YOLO11n-seg':ypeak}
    print('✅ YOLO completado; después sigue Mask R-CNN')


## 5. Medición Mask R-CNN bajo las mismas imágenes y GPU
 Mantiene `min_size=512`, `max_size=768`, pesos y score 0,001 registrados en el 09. Se transfieren máscaras a CPU para igualar el tipo de resultado solicitado.

In [ ]:
def new_maskrcnn():
    m=maskrcnn_resnet50_fpn(weights=None,weights_backbone=None,
        min_size=512,max_size=768,trainable_backbone_layers=3)
    nf=m.roi_heads.box_predictor.cls_score.in_features
    m.roi_heads.box_predictor=FastRCNNPredictor(nf,2)
    in_ch=m.roi_heads.mask_predictor.conv5_mask.in_channels
    m.roi_heads.mask_predictor=MaskRCNNPredictor(in_ch,256,2)
    m.roi_heads.score_thresh=old['mask_score_thresh']
    m.roi_heads.detections_per_img=old['max_dets']
    return m

def collect_mask_stats(model,rgb):
    with torch.inference_mode():
        result=model([to_tensor(rgb).to('cuda:0')])[0]
        masks=(result['masks'][:,0] > old['mask_bin_thresh']).detach().cpu().numpy()
        _=result['scores'].detach().cpu().numpy()
        n=int(np.count_nonzero(masks.reshape(len(masks),-1).max(axis=1)>0)) if len(masks) else 0
        del result
    return n

if EXCEL.exists() and JREPORT.exists():
    print('✅ Resultados finales completos; no se repite Mask R-CNN')
else:
    TEMP=REPORTS/'CERDOS_12_Cache_YOLO.json'
    if not TEMP.exists():raise FileNotFoundError('Falta caché YOLO: primero ejecutar la celda 4')
    cache=json.loads(TEMP.read_text(encoding='utf-8'))
    if cache['protocolo']!=PROTO:raise RuntimeError('La caché YOLO no coincide con el protocolo actual')
    memory_peak={'YOLO11n-seg':cache['vram_mib']}
    model=new_maskrcnn().to('cuda:0')
    state=torch.load(MASK_W,map_location='cpu',weights_only=False)
    model.load_state_dict(state['model'],strict=True)
    del state;model.eval()
    mr,mpeak=timed_model(model,'Mask R-CNN ResNet50-FPN',collect_mask_stats)
    del model;gc.collect();torch.cuda.empty_cache()
    df_raw=pd.DataFrame(cache['datos']+mr)
    memory_peak['Mask R-CNN ResNet50-FPN']=mpeak
    print('✅ Mediciones de ambos modelos completas')


## 6. Informe comparativo, Excel y figura
 Las diferencias en latencia no implican superioridad estadística universal. La memoria CUDA es memoria máxima de PyTorch asignada, **no** consumo eléctrico, consumo total del equipo ni VRAM reservada por CUDA.

In [ ]:
assert set(df_raw['modelo'])=={'YOLO11n-seg','Mask R-CNN ResNet50-FPN'}
assert len(df_raw)==72*2*2,'Se esperaban 288 mediciones de latencia'
assert np.isfinite(df_raw['latencia_ms']).all() and (df_raw['latencia_ms']>0).all()
summary=[]
for name,g in df_raw.groupby('modelo'):
    summary.append({'modelo':name,'gpu_real':GPU,'n_imagenes_unicas':int(g.imagen.nunique()),
        'repeticiones':PROTO['repeticiones'],'observaciones':int(len(g)),
        'latencia_mediana_ms':float(g.latencia_ms.median()),
        'latencia_media_ms':float(g.latencia_ms.mean()),
        'latencia_p95_ms':float(np.quantile(g.latencia_ms,.95)),
        'imagenes_segundo_efectivas':float(len(g)/(g.latencia_ms.sum()/1000)),
        'vram_peak_mib_pytorch':float(memory_peak[name]),
        'mascaras_medias_por_imagen':float(g.mascaras_devueltas.mean())})
df_sum=pd.DataFrame(summary)
df_date=df_raw.groupby(['modelo','fecha'],as_index=False).agg(
    mediciones=('latencia_ms','size'),mediana_ms=('latencia_ms','median'),
    media_ms=('latencia_ms','mean'),p95_ms=('latencia_ms',lambda s:float(np.quantile(s,.95))))
notes=[
    'El tiempo medido omite lectura desde Drive porque las 72 imágenes se precargaron en RAM.',
    'Incluye procesamiento pre/modelo/post y transferencia de máscaras a CPU, no escritura en disco.',
    'CUDA synchronize antes y después de cada imagen; 5 calentamientos y 2 repeticiones.',
    'Las dos canalizaciones tienen tamaños internos, umbrales, arquitecturas y presupuestos distintos.',
    'Pico de memoria PyTorch CUDA asignada, incluyendo el modelo; no equivale a VRAM física total.',
    'La muestra procede exclusivamente de validación; no se reevalúa ni tunea el conjunto test.',
    'Los tiempos son exclusivos de la GPU y del entorno reportados, no extrapolables a producción.',
    'No se mide energía, potencia, CPU total ni persistencia de identidades en video.',
]
with pd.ExcelWriter(EXCEL,engine='openpyxl') as writer:
    df_sum.to_excel(writer,'Resumen',index=False)
    df_date.to_excel(writer,'Por_Fecha',index=False)
    df_raw.to_excel(writer,'Latencias_Individuales',index=False)
    pd.DataFrame([dict((k,json.dumps(v,ensure_ascii=False) if isinstance(v,(list,dict)) else v)
                       for k,v in PROTO.items())]).to_excel(writer,'Protocolo',index=False)
    pd.DataFrame({'limitaciones':notes}).to_excel(writer,'Limitaciones',index=False)

fig,ax=plt.subplots(figsize=(10,5))
names=df_sum['modelo'].tolist()
ax.bar(names,df_sum['latencia_mediana_ms'])
ax.set(title=f'Latencia mediana por imagen en {GPU} (validación)',ylabel='Milisegundos / imagen')
for i,x in enumerate(df_sum.latencia_mediana_ms):ax.text(i,x,f'{x:.1f} ms',ha='center',va='bottom')
fig.tight_layout();fig.savefig(FIG,dpi=180,bbox_inches='tight');plt.show()

report={'etapa':'CERDOS_12_eficiencia_inferencia','fecha_utc':datetime.now(timezone.utc).isoformat(),
    'protocolo':PROTO,'resumen':summary,'picos_vram_mib':memory_peak,'limitaciones':notes}
JREPORT.write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')
print('✅ Resumen de eficiencia, medición de hardware real:')
display(df_sum)
print('✅ Excel:',EXCEL)


## 7. Exportar ZIP, guardar en Drive y descargar al computador

In [ ]:
from google.colab import files
ZIP=EXPORT/'CERDOS_12_Eficiencia_Computacional_Resultados.zip'
entries=[EXCEL,FIG,JREPORT,PFILE]
assert all(p.is_file() for p in entries)
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in entries:z.write(p,arcname=p.name)
with zipfile.ZipFile(ZIP) as z:assert z.testzip() is None,'ZIP inválido'
print('✅ ZIP preparado:',ZIP)
files.download(str(ZIP))


**Siguiente:** compartir ambos ZIP (11 y 12) para revisar si los hallazgos geométricos y de eficiencia añaden una contribución independiente o deben integrarse en el primer artículo. Sin nuevos datos de video, este análisis **no** mide confiabilidad temporal ni seguimiento individual.